In [ ]:
import pandas as pd

arz   = pd.read_csv("cikti/arz_ozeti.csv")
prof  = pd.read_csv("cikti/gercek_haftaici_profil.csv")
talep = pd.read_csv("cikti/talep_final.csv")
dol   = pd.read_csv("cikti/doluluk.csv")
oner  = pd.read_csv("cikti/oneri_sefer_artir.csv")

for ad, df in [("arz_ozeti", arz), ("gercek_haftaici_profil", prof),
               ("talep_final", talep), ("doluluk", dol), ("oneri_sefer_artir", oner)]:
    print(f"{ad:<24} {len(df):>7} satır   sütunlar: {list(df.columns)}")

In [ ]:
ham = pd.read_csv("cikti/sefer_ham_20260806.csv")

print("boyut:", ham.shape)
print("sütunlar:", ham.columns.tolist())
print("\nSYON değerleri:")
print(ham.SYON.value_counts())

ham.head(10)

In [ ]:
yon = ham.groupby(["SHATKODU", "SYON"]).size().unstack(fill_value=0)

print("toplam hat        :", len(yon))
print("sadece G olan hat :", (yon.D == 0).sum())
print("sadece D olan hat :", (yon.G == 0).sum())

yon["fark"] = (yon.G - yon.D).abs()
yon["oran"] = yon.fark / (yon.G + yon.D)
print("dengeli hat (%10'dan az fark):", (yon.oran <= 0.10).sum())

yon.sort_values("fark", ascending=False).head(10)

In [ ]:
# A) sadece G'si olanlar gerçekten halka hat mı? isimlerine bakalım
tek_yon = yon[yon.D == 0].index
print(ham[ham.SHATKODU.isin(tek_yon)][["SHATKODU","HATADI"]]
      .drop_duplicates().head(15).to_string(index=False))

# B) 97 hattında güzergâh varyantları ne diyor?
print("\n--- 97 hattı ---")
print(ham[ham.SHATKODU == "97"]
      .groupby(["SGUZERAH","SYON"]).size().to_string())

In [ ]:
tek_yon = set(yon[yon.D == 0].index.astype(str))

hi = dol[dol.gun_tipi == "WEEKDAYS"].copy()
en = hi.sort_values("doluluk", ascending=False).drop_duplicates("hat")
en["tek_yon"] = en.hat.astype(str).isin(tek_yon)

ust = en[en.doluluk > 110]
print("kapasite üstü hat        :", len(ust))
print("bunlardan tek yönlü olan :", ust.tek_yon.sum())
print("\nmedyan doluluk:")
print(en.groupby("tek_yon").doluluk.median())


In [ ]:
for h in ["19A", "62G"]:
    k = dol[(dol.hat.astype(str) == h) & (dol.gun_tipi == "WEEKDAYS")]
    print(f"\n=== {h} ===  gün toplamı: {k.yolcu_gunluk.sum():,.0f} yolcu / "
          f"{k.sefer_sayisi.sum():,.0f} sefer")
    print(k[["saat","yolcu_gunluk","sefer_sayisi","doluluk"]].to_string(index=False))